# Exploratory Data Analysis (EDA): Customer Churn

In this notebook, we explore the cleaned customer churn dataset to uncover patterns, relationships, and actionable business insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Load cleaned data
df = pd.read_csv('../data/processed/cleaned_churn_data.csv')
df.head()

## 1. Churn Distribution & Class Imbalance

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='Churn', palette='Set2')
plt.title('Overall Churn Distribution')
plt.ylabel('Number of Customers')

total = len(df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    x = p.get_x() + p.get_width() / 2 - 0.1
    y = p.get_y() + p.get_height() + 50
    ax.annotate(percentage, (x, y))

plt.show()

### Finding 1: Churn Distribution
**1. What the chart shows:** The target variable `Churn` is imbalanced. ~26.5% of customers churned, while 73.5% remained.

**2. Why the finding matters:** An imbalanced dataset means accuracy is not a reliable evaluation metric for machine learning. A model that predicts "No Churn" every time would be 73.5% accurate but completely useless for identifying at-risk customers.

**3. What business action it could suggest:** The baseline churn rate of 26.5% sets our benchmark. Any retention strategy must prove it can significantly lower this baseline rate.

**4. Caveats:** This is historical data. The current churn rate may differ if market conditions or competitor offerings have changed.

## 2. Tenure and Churn

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='tenure', hue='Churn', multiple='stack', bins=36, palette='Set2')
plt.title('Customer Churn by Tenure (Months)')
plt.xlabel('Tenure (Months)')
plt.ylabel('Number of Customers')
plt.show()

### Finding 2: Tenure & Churn Relationship
**1. What the chart shows:** A massive spike in churn occurs during the first few months (0-5 months) of service. Customers with high tenure (60-70+ months) rarely churn.

**2. Why the finding matters:** Customer loyalty increases over time, or alternatively, unsatisfied customers leave early. The onboarding and early lifecycle phases are critical failure points.

**3. What business action it could suggest:** Implement a targeted "First 90 Days" retention program. Offer proactive support, onboarding check-ins, or introductory discounts that bridge them past the critical early drop-off period.

**4. Caveats:** Correlation is not causation. Customers might stay longer *because* they are happy with the product, not just because time has passed.

## 3. Contract Type and Churn

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Contract', hue='Churn', palette='Set2')
plt.title('Churn by Contract Type')
plt.ylabel('Number of Customers')
plt.show()

### Finding 3: Contract Type & Churn
**1. What the chart shows:** Month-to-month contracts have an overwhelmingly high churn volume and rate compared to one-year and two-year contracts.

**2. Why the finding matters:** Customers without long-term commitments face no friction when deciding to leave.

**3. What business action it could suggest:** Incentivize month-to-month customers to switch to annual contracts by offering a discounted yearly rate or premium feature upgrades.

**4. Caveats:** Forcing long-term contracts might reduce initial sign-ups. The high churn in month-to-month might be expected and priced into the business model.

## 4. Monthly Charges and Churn

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', common_norm=False, fill=True, palette='Set2')
plt.title('Density of Monthly Charges by Churn Status')
plt.xlabel('Monthly Charges ($)')
plt.show()

### Finding 4: Monthly Charges & Churn
**1. What the chart shows:** Customers who churn tend to have higher monthly charges (density peaks around $70-$105). Customers with very low monthly charges (~$20) rarely churn.

**2. Why the finding matters:** Cost is a significant driver of churn. Customers paying a premium may feel they are not getting sufficient value.

**3. What business action it could suggest:** Review the pricing strategy for premium tiers. Consider bundling value-add services for high-paying customers to increase their perceived value without lowering the price.

**4. Caveats:** Higher charges often correlate with month-to-month contracts or specific add-ons (like Fiber Optic). We must control for these variables before concluding that price alone causes churn.

## 5. Payment Method and Churn

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='PaymentMethod', hue='Churn', palette='Set2')
plt.title('Churn by Payment Method')
plt.xticks(rotation=15)
plt.ylabel('Number of Customers')
plt.show()

### Finding 5: Payment Method & Churn
**1. What the chart shows:** Customers paying via Electronic check have a vastly disproportionate churn rate compared to automated methods (Credit card, Bank transfer).

**2. Why the finding matters:** Manual payment methods involve active friction every month. An electronic check requires the customer to consciously authorize payment, reminding them of the expense and giving them an opportunity to cancel.

**3. What business action it could suggest:** Push customers toward automated billing (autopay). Offer a small monthly discount (e.g., $5 off) for enrolling in credit card autopay.

**4. Caveats:** Demographic differences could explain this. Perhaps older or less tech-savvy customers use Electronic checks and are churning for other service-related reasons.

## 6. Correlation Matrix

In [ ]:
# Convert Churn to binary for correlation matrix
df_corr = df.copy()
df_corr['Churn_Binary'] = df_corr['Churn'].map({'Yes': 1, 'No': 0})

# Select numerical features
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_Binary']
corr = df_corr[num_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f')
plt.title('Correlation Matrix of Numerical Features')
plt.show()

### Finding 6: Numerical Correlations
**1. What the chart shows:** Tenure is negatively correlated with Churn (-0.35), while MonthlyCharges has a slight positive correlation (0.19).

**2. Why the finding matters:** This confirms quantitatively what the bar charts showed qualitatively: longer tenure reduces churn risk, higher prices increase it.

**3. What business action it could suggest:** Use tenure and monthly charges as foundational numerical features in our predictive model, as they clearly have predictive power.

**4. Caveats:** Linear correlation (Pearson) only captures linear relationships. The relationship between MonthlyCharges and Churn may be non-linear (e.g., churn only spikes after a specific price threshold).